# Data Preparation

Notebook ini menyiapkan data tiket mentah menjadi format yang siap dipakai untuk embedding, word cloud, dan clustering. Alurnya mencakup pembersihan teks, normalisasi, pembentukan embedding IndoSBERT, lalu penyimpanan hasil ke file turunan.

## Set Up Environment

Bagian ini menyiapkan lingkungan Python yang dibutuhkan sebelum proses preprocessing dijalankan, termasuk pemasangan library utama yang dipakai di notebook.

### Install Dependencies

Sel jalur ini memasang pustaka yang dipakai untuk pembacaan data, pemrosesan teks, progress bar, dan model embedding IndoSBERT.

In [ ]:
%pip install --upgrade pandas sentence-transformers pandas tqdm urllib3 Sastrawi

### Import Dependencies

Setelah library terpasang, sel ini mengimpor modul yang diperlukan untuk manipulasi data, regular expression, pemrosesan URL, dan stopword bahasa Indonesia.

In [ ]:
import pandas as pd
import numpy as np
import re
import os

from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from urllib.parse import urlparse
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

### Load Model Sentence-BERT (IndoSBERT)

Model IndoSBERT dimuat sebagai encoder utama untuk mengubah tiket teks menjadi vektor embedding semantik.

In [ ]:
model_name = 'denaya/indoSBERT-large'

model = SentenceTransformer(model_name)

### Load Raw Data

Tahap ini membaca data mentah tiket dari CSV, mengambil kolom deskripsi, lalu menyiapkan data awal yang akan dibersihkan.

In [ ]:
data_path = 'datasets/raw_tickets.csv'

df_raw = pd.read_csv(data_path)

df = df_raw[['DESKRIPSI']].dropna()
df.rename(columns={'DESKRIPSI': 'RAW TICKET'}, inplace=True)

df.head()

## Data Preprocessing

Bagian ini mengubah teks mentah menjadi representasi yang lebih seragam dan lebih siap untuk embedding maupun visualisasi kata.

### Preprocessed Data for Sentence-Embedding Model

Subbagian ini fokus pada pembersihan teks yang dipakai sebagai input model embedding, sehingga hasil vektor lebih stabil dan konsisten.

#### Cleansing

Langkah cleansing menghapus karakter yang tidak diperlukan, placeholder tertentu, dan emotikon agar isi tiket menjadi lebih bersih.

In [ ]:
# Remove New Line Special Character
df['CLEANED TICKET'] = df['RAW TICKET'].apply(lambda x: re.sub(r'\n+', '. ', x))

# Remove klien and label isu/kendala
def remove_placeholder(teks):
    if not isinstance(teks, str):
        return teks
    
    # 1a. Tiket berformat label: buang blok klien sampai sebelum label isu/kendala
    pattern_klien_berlabel = r'\b(?:(?:klien|client|clien|lien)[\s:;,\.]*|(?:dashboard trial|dashboard client|dashboard|project)\s*[:;]\s*)(.*?)(?=\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\b)'
    teks = re.sub(pattern_klien_berlabel, '', teks, flags=re.IGNORECASE)
    
    # 1b. Sisa "klien: <nama>" yang mungkin masih ada (tiket tanpa label isu)
    # Hanya buang token "klien:" dan nilai langsung setelahnya sampai titik/koma berikutnya
    pattern_klien_sisa = r'\b(?:klien|client|clien|lien)[\s:;,\.]*[^.!?\n]*?(?=[.!?\n]|$)'
    teks = re.sub(pattern_klien_sisa, '', teks, flags=re.IGNORECASE)
    
    # 2. Hapus label isu/kendala (tidak berubah)
    pattern_isu_detail = r'\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\s*[:;]\s*'
    teks = re.sub(pattern_isu_detail, '', teks, flags=re.IGNORECASE)
    
    # 3. Finalisasi
    teks = teks.strip(' ;:,-')
    teks = re.sub(r'[ \t]+', ' ', teks)
    
    return teks

df['CLEANED TICKET'] = df['CLEANED TICKET'].apply(remove_placeholder)


# Remove Emoticon
def remove_emoticon(teks):
    if not isinstance(teks, str):
        return teks
    # Rentang emotikon umum (emoji, simbol, dll.)
    emoticon_pattern = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF]+'
    return re.sub(emoticon_pattern, '', teks, flags=re.IGNORECASE)

df['CLEANED TICKET'] = df['CLEANED TICKET'].apply(remove_emoticon)


df.head()

#### Normalization

Tahap normalisasi menyeragamkan huruf, mengganti nama akun, email, sapaan, dan URL dengan token yang lebih umum.

In [ ]:
# Case Folding
df['NORMALIZED TICKET'] = df['CLEANED TICKET'].str.lower()

# Replace Account Name and Email Address with placeholder
# Replace Account Name with placeholder "nama orang" (contoh: @username)
df['NORMALIZED TICKET'] = df['NORMALIZED TICKET'].apply(lambda x: re.sub(r'@\w+', 'nama orang', x))

# Replace Email Address with placeholder "email" (contoh: email@domain.com)
df['NORMALIZED TICKET'] = df['NORMALIZED TICKET'].apply(lambda x: re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email', x))

# Replace Greetings with placeholder "sapaan" (contoh: halo, hai, selamat pagi)
greeting_pattern = (
    r'\b('
    # Sapaan berbasis waktu (panjang & pendek)
    r'selamat pagi|selamat siang|selamat sore|selamat malam|'
    r'pagi|siang|sore|malam|'
    # Sapaan umum & informal (termasuk huruf berulang seperti halloo, haloo)
    r'hal+o+|hel+o+|hi|permisi|assalamu\'?alaikum|assalamualaikum|'
    # Sapaan daerah / Sunda
    r'punteun|punten|nuhun|'
    # Frasa permohonan bantuan
    r'mohon dibantu|mohon bantuan(?:nya)?|minta tolong|tolong dibantu|tolong bantuan(?:nya)?|'
    # Ucapan terima kasih dan penutup
    r'terima\s?kasih\s?sebelumnya|terimakasih\s?sebelumnya|'
    r'terima\s?kasih\s?banyak|terimakasih\s?banyak|'
    r'terima\s?kasih|terimakasih|makasih|'
    r'thanks|thank\s?you|guys'
    r')\b'
)
df['NORMALIZED TICKET'] = df['NORMALIZED TICKET'].apply(lambda x: re.sub(greeting_pattern, 'sapaan', x, flags=re.IGNORECASE))

# Replace URL address with it's domain name and 'tautan' placeholder (contoh: https://www.example.com/path -> tautan example)
def parse_url(teks):
    if not isinstance(teks, str):
        return teks
    
    # POLA REGEX BARU: 
    # Menangkap seluruh string URL yang valid (termasuk huruf, angka, strip, titik, dan garis miring)
    pola_url = r'https?://[\w\-\.\/\?\&\=\%]+'
    
    def ekstrak_domain(match):
        url = match.group(0)
        
        # Mencegah titik atau koma di akhir kalimat ikut terbaca sebagai bagian URL
        if url.endswith('.') or url.endswith(','):
            url = url[:-1]
            
        try:
            # urlparse akan membedah URL. 
            # Contoh: dari "https://megapolitan.kompas.com/..." kita ambil "megapolitan.kompas.com"
            netloc = urlparse(url).netloc
            
            # Pecah berdasarkan titik
            parts = netloc.split('.')
            
            # LOGIKA PENCARIAN NAMA DOMAIN UTAMA:
            # Kasus 1: Domain Indonesia 3 tingkat (contoh: news.detik.co.id -> ambil 'detik')
            if len(parts) >= 3 and parts[-2] in ['co', 'go', 'ac', 'or', 'sch', 'my']:
                domain_utama = parts[-3]
                
            # Kasus 2: Domain standar dengan/tanpa subdomain (contoh: megapolitan.kompas.com -> ambil 'kompas')
            elif len(parts) >= 2:
                domain_utama = parts[-2]
                
            # Kasus 3: Fallback jika format URL tidak biasa
            else:
                domain_utama = parts[0]
                
            # Jika domain utama tertangkap sebagai 'www', ambil kata setelahnya
            if domain_utama == 'www' and len(parts) >= 2:
                domain_utama = parts[-1]
                
            return f"tautan {domain_utama.lower()}"
            
        except Exception:
            return "tautan" # Fallback jika terjadi error parsing
            
    # Terapkan re.sub menggunakan fungsi ekstrak_domain
    teks = re.sub(pola_url, ekstrak_domain, teks)
    
    return teks

# Terapkan fungsi ke kolom yang sudah diproses sebelumnya
df['NORMALIZED TICKET'] = df['NORMALIZED TICKET'].apply(parse_url)

df.head()

#### Save Results to CSV

Hasil pembersihan dan normalisasi disimpan ke CSV agar bisa dipakai ulang pada notebook clustering tanpa menghitung ulang preprocessing.

In [ ]:
# Simpan hasil akhir ke CSV baru
output_path = 'datasets/cleaned_tickets.csv'

df_result = df[['RAW TICKET', 'CLEANED TICKET', 'NORMALIZED TICKET']].copy()
df_result.rename(columns={'PARSED URL': 'CLEANED TICKET'}, inplace=True)

df_result.to_csv(output_path, index=False)

### Sentence-Embedding

Bagian ini mengubah teks hasil preprocessing menjadi embedding numerik yang dapat digunakan oleh algoritma clustering.

#### Implements Sliding Windows

Metode sliding window dipakai untuk menangani tiket yang panjang agar seluruh teks tetap bisa di-encode oleh model.

In [ ]:
tokenizer = model.tokenizer

def get_sliding_window_embedding(teks, model, max_length=256, stride=128):
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = 100_000
    tokens = tokenizer.encode(teks, add_special_tokens=False)
    tokenizer.model_max_length = original_max_length

    # Jika pendek, langsung encode
    if len(tokens) <= max_length - 2:
        return model.encode(teks)

    window_size = max_length - 2  # ruang untuk [CLS] dan [SEP]
    step = window_size - stride    # seberapa jauh window bergeser tiap iterasi

    chunk_embeddings = []
    start = 0

    while start < len(tokens):
        end = min(start + window_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunk_emb = model.encode(chunk_text)
        chunk_embeddings.append(chunk_emb)
        
        if end == len(tokens):  # sudah sampai akhir
            break
        start += step

    return np.mean(chunk_embeddings, axis=0)

#### Embed Sentence

Sel ini menjalankan embedding pada setiap tiket dan menyimpan hasil vektornya ke dataframe keluaran.

In [ ]:
# Mengaktifkan ekstensi pandas dari tqdm untuk memunculkan progress bar
tqdm.pandas(desc="Proses Embedding IndoSBERT")

# 1. Mencegah Error: Pastikan tidak ada data kosong (NaN) akibat proses pembersihan sebelumnya
df_result['NORMALIZED TICKET'] = df_result['NORMALIZED TICKET'].fillna("").astype(str)

# 2. Ekstraksi Embedding dengan Sliding Window
df_result['EMBEDDING'] = df_result['NORMALIZED TICKET'].progress_apply(
    lambda teks: get_sliding_window_embedding(
        teks=teks, 
        model=model, 
        max_length=256, 
        stride=128
    )
)

# Menampilkan 5 baris pertama untuk memastikan kolom EMBEDDING sudah terbentuk
df_result.head()

### Preprocessed Data for Word Cloud

Selain untuk embedding, teks juga dipersiapkan untuk word cloud dengan menyaring kata-kata noise yang terlalu umum.

In [ ]:
keywords_to_keep = {
    'belum', 'tidak', 'kurang', 'bisa', 'semua', 'atas', 'bawah', 
    'masuk', 'keluar', 'hilang', 'kosong', 'gagal', 'salah'
}

khas_tiket_noise = {
    'mas', 'mba', 'om', 'tim', 'it', 'mohon', 'dibantu', 'punteun', 'tolong', 
    'cek', 'bantu', 'halo', 'min', 'admin', 'dari', 'ke', 'perihal', 'tautan',
    'siang', 'pagi', 'sore', 'malam', 'selamat', 'terima', 'kasih', 'sebelumnya',
    'ya', 'yaa', 'kah', 'lah', 'pun', 'terimakasih', 'salam', 'sapaan', 'nama', 'orang',
    'email', 'domain', 'com', 'kak', 'guys', 'klien', 'client', 'user', 'bantuannya', 
    'makasih', 'nuhun', 'punten', 'maaf', 'izin', 'tanya', 'informasi', 'info', 'link', 
    'url', 'kolom', 'kendala', 'isu', 'issue', 'muncul', 'sesuai', 'terkait', 'dicek', 
    'update', 'tanggal', 'jam', 'periode', 'nya', 'yg', 'ga', 'gak', 'gaada', 'gabisa', 
    'udah', 'udh', 'buat', 'kalo', 'kalau', 'tapi', 'tetap', 'ttp', 'sih', 'kok', 'kan', 
    'dong', 'deh', 'mah', 'aja', 'terus', 'data', 'mati'
}

factory = StopWordRemoverFactory()
stopword_indonesia = factory.get_stop_words()

stopwords = set(stopword_indonesia) - keywords_to_keep
stopwords.update(khas_tiket_noise)

In [ ]:
def normalize_token(word):
    cleaned = re.sub(r'^\W+|\W+$', '', word.lower())
    return cleaned


df_result['WORDCLOUD TICKET'] = df_result['NORMALIZED TICKET'].apply(
    lambda x: ' '.join(
        [word for word in x.split() if normalize_token(word) and normalize_token(word) not in stopwords]
    )
)


df_result.head()

In [ ]:
# Simpan hasil akhir ke CSV baru
output_path = 'datasets/preprocessed_output.csv'

df_result.to_csv(output_path, index=False)

#### Save to Pickle

Data akhir disimpan ke pickle agar notebook clustering dapat memuatnya dengan cepat bersama embedding yang sudah terbentuk.

In [ ]:
output_path = 'datasets/tickets.pkl'

df_result.to_pickle(output_path)